In [1]:
import os
from summarization.prune import prune_graph_pipeline, PruneGraph
# from summarization.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )
device = "cuda"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
from summarization.token_attribution import get_token_attribution_from_graph

for normalize_method in ['softmax', 'sparsemax', 'entmax15', 'relu_l1']:    
    token_weights = get_token_attribution_from_graph(
        graph_path = "temp_graph_files/clt-hp/clt-hp-p05-fact-the-language-spoken-20260424-171104-257980.json",
        model_name = "google/gemma-2-2b",
        normalize_method = normalize_method,
        device = "cuda"
    )
    print(token_weights)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:10, 10.84s/it]               


tensor([4.8364e-04, 4.8364e-04, 2.4635e-03, 2.0126e-02, 6.5109e-03, 1.8393e-03,
        9.5954e-04, 1.0395e-03, 2.9242e-04, 6.8976e-04, 3.6327e-03, 9.5885e-01,
        2.6303e-03])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])
tensor([0.0000, 0.0000, 0.0727, 0.1665, 0.1161, 0.0596, 0.0306, 0.0342, 0.0000,
        0.0158, 0.0900, 0.3389, 0.0756])


In [17]:
# Print token number 11430 of the vocab of the tokenizer
token_str = tokenizer.decode([11430])
print(f"Token id 11430: '{token_str}'")

Token id 11430: ' Japanese'


In [10]:
explainer = shap.Explainer(model,tokenizer)

In [11]:
prompts = ['Fact: The language spoken in the country whose capital is Tokyo is']

In [12]:
shap_values = explainer(prompts)

In [13]:
shap.plots.text(shap_values)

In [14]:
print(shap_values)

.values =
array([[[ 1.78116298e-06],
        [ 8.51497794e-01],
        [-4.86745747e-01],
        [ 1.62801018e+00],
        [ 3.72844004e+00],
        [ 2.59989587e+00],
        [ 1.33582383e+00],
        [ 6.85110150e-01],
        [ 7.65128036e-01],
        [-5.03143328e-01],
        [ 3.55010058e-01],
        [ 2.01639305e+00],
        [ 7.59215119e+00],
        [ 1.69349370e+00]]])

.base_values =
array([[-22.91878813]])

.data =
(array(['', 'Fact', ':', ' The', ' language', ' spoken', ' in', ' the',
       ' country', ' whose', ' capital', ' is', ' Tokyo', ' is'],
      dtype=object),)


In [16]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()[1:]) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[1.13271423e-03 2.97117732e-04 2.46238185e-03 2.01168742e-02
 6.50788688e-03 1.83848665e-03 9.59089468e-04 1.03898780e-03
 2.92285447e-04 6.89443169e-04 3.63101413e-03 9.58404695e-01
 2.62902390e-03]


## Generate graph if not exist

In [ ]:
# from generate_new_graphs import download_graph
# download_graph(
#     prompt=prompts[0],
#     slug="austin_plt",
#     source_set = 'gemmascope-transcoder-16k',
#     out_path="temp_graph_files/austin_plt.json",
#     node_threshold = 0.95,
#     edge_threshold = 0.98,
# )

requesting graph for slug='austin_plt' prompt='Fact: The capital of the state containing Dallas is...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/austin_plt.json
saved austin_plt -> temp_graph_files/austin_plt.json (nodes=5387)


## ASG Pipeline

### Prune

In [7]:
from summarization.prune import prune_graph_pipeline
from summarization.prune import load_prune_graph
# prune_graph = load_prune_graph('temp_graph_files/clt-hp/clt-hp-p05-fact-the-language-spoken-20260424-171104-257980.json')
token_weights = [0, 0.0000, 0.0000, 0.0727, 0.1665, 0.1161, 0.0596, 0.0306, 0.0342, 0.0000,
        0.0158, 0.0900, 0.3389, 0.0756]
# 1) Build prune_graph as usual
prune_graph = prune_graph_pipeline(
    json_path="temp_graph_files/clt-hp/clt-hp-p05-fact-the-language-spoken-20260424-171104-257980.json",
    token_weights = token_weights,
    logit_weights="target",
    node_influence_threshold=0.7,
    node_relevance_threshold=0.7,
    edge_influence_threshold=0.95,
    edge_relevance_threshold=0.95,
    keep_all_tokens_and_logits=False,
    filter_act_density=True,
)


In [8]:
num_nodes = prune_graph.num_nodes
num_edges = (prune_graph.pruned_adj != 0).sum().item()
num_edges_positive = (prune_graph.pruned_adj > 0).sum().item()
num_edges_negative = (prune_graph.pruned_adj < 0).sum().item()
print(f"Number of nodes: {num_nodes}")
print(f"Total nonzero edges: {num_edges}")
print(f"Edges with weight > 0: {num_edges_positive}")
print(f"Edges with weight < 0: {num_edges_negative}")

Number of nodes: 27
Total nonzero edges: 123
Edges with weight > 0: 74
Edges with weight < 0: 49


In [9]:
for node_id in prune_graph.kept_ids:
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_influence[prune_graph.kept_ids.index(node_id)].item(), prune_graph.node_relevance[prune_graph.kept_ids.index(node_id)].item())


16_59225_13 Japanese 0.015179392881691456 0.012604701332747936
16_87097_13 Languages 0.005267545115202665 0.010529029183089733
16_92378_13 Asian countries/languages 0.008203539997339249 0.012245498597621918
18_40581_13 Japan 0.022769194096326828 0.020040424540638924
18_54160_13 Japanese/Japan 0.009618476033210754 0.012777051888406277
18_62070_13 languages 0.007334739901125431 0.019742298871278763
19_36461_13 Japan 0.019269336014986038 0.01699046604335308
19_45624_13 technical, code, and science 0.008463677018880844 0.017053602263331413
19_62855_13 languages 0.021965287625789642 0.022892024368047714
20_17935_12 Japanese 0.008889667689800262 0.024884991347789764
20_54580_12 Japan 0.022271277382969856 0.022492211312055588
20_17935_13 Japanese 0.02283976599574089 0.026930388063192368
20_54580_13 Japan 0.017973950132727623 0.024530161172151566
22_19177_13 Japanese writing 0.02126229554414749 0.028175322338938713
22_23892_13 with 0.008987545035779476 0.023790588602423668
23_50752_13 Diverse 

## Cluster

In [54]:
from summarization.prune import save_prune_graph, load_prune_graph
# save_prune_graph(prune_graph, "subgraph/austin_clt_clean_5_9_5.pt")
prune_graph = load_prune_graph("subgraph/austin_clt_clean_5_9_5.pt")

In [11]:
from summarization.cluster import cluster_graph, cluster_graph_with_labels
from summarization.auto_grouping import find_best_k
# 2) Auto-select k
best_k, sweep = find_best_k(
    prune_graph,
    max_layer_span=4,
    mediation_penalty=0.1,               # how much F(phi) affects total
)

print("best_k:", best_k)
print("best score:", sweep[best_k]["total"] if best_k in sweep else None)

# 3) Run final clustering with best_k
supernodes = cluster_graph_with_labels(
    prune_graph,
    target_k=best_k,
    max_layer_span=4,
    mediation_penalty=0.1,
    enforce_dag=True,
)

best_k: 3
best score: 0.824828365033692


In [12]:
supernodes

[['cluster_0', '16_59225_13', '16_87097_13', '16_92378_13'],
 ['cluster_1',
  '18_40581_13',
  '18_54160_13',
  '19_36461_13',
  '20_54580_12',
  '20_54580_13'],
 ['cluster_2', '18_62070_13', '19_45624_13', '19_62855_13'],
 ['cluster_3',
  '20_17935_12',
  '20_17935_13',
  '22_19177_13',
  '22_23892_13',
  '23_50752_13']]

In [14]:
for supernode in supernodes:
    print(supernode[0])
    for node_id in supernode[1:]:
        print(node_id, prune_graph.attr[node_id].get("clerp", ""))

cluster_0
16_59225_13 Japanese
16_87097_13 Languages
16_92378_13 Asian countries/languages
cluster_1
18_40581_13 Japan
18_54160_13 Japanese/Japan
19_36461_13 Japan
20_54580_12 Japan
20_54580_13 Japan
cluster_2
18_62070_13 languages
19_45624_13 technical, code, and science
19_62855_13 languages
cluster_3
20_17935_12 Japanese
20_17935_13 Japanese
22_19177_13 Japanese writing
22_23892_13 with
23_50752_13 Diverse information and documentation


### Visualize on the web

In [66]:
import json
from api import save_subgraph

status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug=prune_graph.metadata["slug"],                       # parent graph slug
    displayName="austin-baseline",     # subgraph name shown on Neuronpedia
    pinnedIds=prune_graph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.8,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

{'Content-Type': 'application/json', 'x-api-key': 'sk-np-Vi0OcQl8zNm9jC7nyGRYfwssvSakMyrPjV675uEWIuU0'}
status: 200
{'success': True, 'subgraphId': 'cmo5mvtbc000013to8x5c7pqy'}


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
Prune: faithfulness, completeness
Kmeans: node profile (layer, influence, relevance, (activation))
Spectral: adj == affinity matrix
hyperparams: 
Ablation: 